# Week 1, Lab 5 — Mini-project: research & summarize (no framework)

Checkpoint: if you can build this, you are ready for Weeks 2–6.

**Brief:** A user gives a topic. The agent looks up local facts (and can do math if needed), then returns a 4-bullet summary plus a one-line takeaway.


## 1. Setup


In [ ]:
WEEK = 'Week 1'
LAB = 'Lab 5 — mini-project'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn pydantic
else:
    %pip install -q ollama pydantic


## 2. Build your agent

Reuse structured output + tools + a ReAct loop. Keep the user-facing answer **plain language**, not JSON.


In [ ]:
from pydantic import BaseModel

class Summary(BaseModel):
    bullets: list[str]
    takeaway: str

SYSTEM = """Research assistant. Use lookup_fact for topics you are unsure about.
You may call calculator if numbers appear.
Each turn output ONLY JSON:
{"name": "lookup_fact"|"calculator"|"today_date", "arguments": {...}}
or {"final": true, "bullets": ["...", "..."], "takeaway": "..."}
Need 3-5 bullets. Do not invent facts not in the tool results.
"""

def research(topic: str, max_steps: int = 8) -> Summary:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Research and summarize: {topic}"},
    ]
    observations = []
    for _ in range(max_steps):
        reply = local_chat(messages, max_new_tokens=220, temperature=0.15)
        obj = extract_json_object(reply) or {}
        if obj.get("final") or "bullets" in obj:
            try:
                return Summary(bullets=obj.get("bullets") or [], takeaway=obj.get("takeaway") or "")
            except Exception:
                pass
        name, args = obj.get("name"), obj.get("arguments") or {}
        fn = {"lookup_fact": lookup_fact, "calculator": calculator, "today_date": today_date}.get(name)
        result = fn(**args) if fn and args else (fn() if fn else f"bad tool {obj}")
        observations.append(result)
        messages += [
            {"role": "assistant", "content": reply},
            {"role": "user", "content": f"OBSERVATION: {result}\nKnown observations: {observations}"},
        ]
    return Summary(bullets=observations[:4] or ["No facts gathered."], takeaway="Loop ended early.")

result = research("MCP and Ollama")
print("TAKEAWAY:", result.takeaway)
for b in result.bullets:
    print("-", b)


## 3. Rubric (for you and your instructor)

- [ ] Uses at least one tool (not just the model's memory)
- [ ] Final answer is structured (`Summary`)
- [ ] Stops (no infinite loop)
- [ ] Works on Colab **or** local Ollama

## 4. Stretch

Swap `lookup_fact` for 5 of your own course notes in a dict. That is a baby RAG without embeddings.

**Next week:** OpenAI Agents SDK pointed at the same local model.
